# Predicting behaviour from neural geometry needs a proper metric

We simulate a cohort of subjects, each with a **neural geometry** and a **behaviour**,
and predict behaviour from geometry using three notions of dissimilarity:

| | symmetric? | triangle inequality? |
|---|---|---|
| **Procrustes** shape distance | yes | yes |
| **linear predictivity**, $1-R^2$ of a cross-validated linear map | **no** | no |
| **symmetrized predictivity**, $(D + D^\mathsf{T})/2$ | yes | **no** |

Nearest-neighbour regression only needs the *ordering* of distances, so it is a direct
test of whether a dissimilarity behaves like a distance.

Run top to bottom; takes about two minutes.

In [1]:
import itertools

import matplotlib.pyplot as plt
import numpy as np
import seaborn as sns
from joblib import Parallel, delayed
from netrep.metrics import LinearMetric
from sklearn.decomposition import PCA
from sklearn.linear_model import RidgeCV
from sklearn.metrics import r2_score
from sklearn.model_selection import KFold

# ---------------------------------------------------------------- the cohort
N_SUBJECTS   = 30
N_NEURONS    = 60
N_HARMONICS  = 4       # the stimulus tuning is band-limited to this many harmonics
N_PCS        = 10      # every dissimilarity sees the same reduced data
N_NEIGHBOURS = 3       # k for the nearest-neighbour regression
N_SEEDS      = 50      # independent cohorts

# ---------------------------------------------------------------- the experiment
N_STIMULI    = 60
N_OTHER_VARS = 5       # other task variables, two levels each
MIXING       = 0.22    # how strongly the other variables are encoded
NOISE_VAR    = 0.1     # trial-to-trial variance assumed for Fisher information
RIDGE_ALPHAS = np.logspace(-4, 4, 17)

# Three tuning-width types, set by the weight given to each harmonic.  All three
# span the same set of dimensions, so any two differ by an invertible linear
# reweighting -- precisely the transformation a linear predictivity score is
# designed not to see.
TUNING_TYPES = {
    "broad tuning": np.array([1.00, 0.15, 0.05, 0.02]),
    "intermediate": np.array([1.00, 0.70, 0.35, 0.15]),
    "sharp tuning": np.array([1.00, 0.95, 0.85, 0.70]),
}

# The measurement design is additive (one factor at a time): first the stimulus is
# varied with the other variables at baseline, then the other variables are varied
# at a fixed stimulus.  Every subject is measured at the same conditions.
STIMULI       = np.linspace(-np.pi, np.pi, N_STIMULI, endpoint=False)
OTHER_LEVELS  = np.array(list(itertools.product([-1, 1], repeat=N_OTHER_VARS)))
COND_STIMULUS = np.concatenate([STIMULI, np.zeros(len(OTHER_LEVELS))])
COND_OTHER    = np.vstack([np.zeros((N_STIMULI, N_OTHER_VARS)), OTHER_LEVELS])
N_CONDITIONS  = len(COND_STIMULUS)
STIMULUS_ONLY = slice(0, N_STIMULI)     # conditions where only the stimulus varies

# ---------------------------------------------------------------- presentation
METRICS      = ["procrustes", "predictivity", "symmetrized"]
METRIC_LABEL = {"procrustes": "Procrustes",
                "predictivity": "predictivity $D$",
                "symmetrized": r"$D + D^\mathsf{T}$"}
METRIC_COLOR = {"procrustes": "#eb6834",
                "predictivity": "#0b0b0b",
                "symmetrized": "#8a8984"}

print(f"{N_CONDITIONS} conditions = {N_STIMULI} stimulus "
      f"+ {len(OTHER_LEVELS)} combinations of {N_OTHER_VARS} other task variables")

92 conditions = 60 stimulus + 32 combinations of 5 other task variables


## 1. The model

Subjects differ in two independent respects:

1. **tuning width** — how sharply neurons are tuned to the stimulus. This determines
   behaviour, via the Fisher information of the stimulus code.
2. **how many of the other task variables they encode** (0–5). Irrelevant to behaviour.

Each cohort is built three times, differing only in how those other variables are
encoded at the level of single neurons.

In [2]:
def stimulus_tuning(weights, preferred, stimuli):
    """Peak-normalised periodic tuning of each neuron to the stimulus."""
    rate = sum(weights[h - 1] * np.cos(h * (stimuli[None, :] - preferred[:, None]))
               for h in range(1, N_HARMONICS + 1))
    return rate / np.abs(rate).max()


def encode_segregated(weights, n_vars, rng, n_stimulus_neurons=30):
    """No mixing: disjoint groups of neurons for the stimulus and each variable."""
    rate = np.zeros((N_NEURONS, N_CONDITIONS))
    rate[:n_stimulus_neurons] = stimulus_tuning(
        weights, rng.uniform(-np.pi, np.pi, n_stimulus_neurons), COND_STIMULUS)
    if n_vars:                     # np.array_split rejects zero sections
        spare = np.arange(n_stimulus_neurons, N_NEURONS)
        for v, group in enumerate(np.array_split(spare, n_vars)):
            rate[group] = MIXING * np.outer(rng.uniform(-1, 1, len(group)),
                                            COND_OTHER[:, v])
    return rate


def encode_linear_mixed(weights, n_vars, rng):
    """Additive mixed selectivity: every neuron responds to the stimulus and to
    each variable, so each variable adds one dimension."""
    rate = stimulus_tuning(weights, rng.uniform(-np.pi, np.pi, N_NEURONS),
                           COND_STIMULUS)
    for v in range(n_vars):
        rate = rate + MIXING * np.outer(rng.uniform(-1, 1, N_NEURONS),
                                        COND_OTHER[:, v])
    return rate


def encode_nonlinear_mixed(weights, n_vars, rng):
    """Conjunctive: the other variables gain-modulate the stimulus tuning."""
    rate = stimulus_tuning(weights, rng.uniform(-np.pi, np.pi, N_NEURONS),
                           COND_STIMULUS)
    gain = np.ones_like(rate)
    for v in range(n_vars):
        gain = gain + MIXING * np.outer(rng.uniform(-1, 1, N_NEURONS),
                                        COND_OTHER[:, v])
    return rate * gain


SCHEMES = {"segregated": encode_segregated,
           "linear mixed": encode_linear_mixed,
           "nonlinear mixed": encode_nonlinear_mixed}

# Behaviour is read out from a large independent pool of neurons -- the asymptotic
# value, not an estimate from the N_NEURONS we "record".
POOL_PREFERRED = np.random.RandomState(0).uniform(-np.pi, np.pi, 2000)


def acuity(weights):
    """log Fisher information per neuron: how finely the population can
    discriminate nearby stimuli.  Larger for sharper tuning."""
    tuning = stimulus_tuning(weights, POOL_PREFERRED, STIMULI)
    slope = np.gradient(tuning, STIMULI[1] - STIMULI[0], axis=1)
    return np.log(np.mean(slope ** 2) / NOISE_VAR)


def make_cohort(seed):
    """One cohort of N_SUBJECTS subjects, encoded under every scheme.

    This is the only place a cohort is built, so the single-cohort figure and the
    across-cohort summary can never drift apart.
    """
    rng = np.random.RandomState(seed)
    types = np.repeat(np.arange(len(TUNING_TYPES)), N_SUBJECTS // len(TUNING_TYPES))
    profiles = list(TUNING_TYPES.values())
    weights = np.array([profiles[t] * rng.uniform(0.9, 1.1, N_HARMONICS)
                        for t in types])
    n_vars = rng.randint(0, N_OTHER_VARS + 1, N_SUBJECTS)
    behaviour = np.array([acuity(w) for w in weights]) \
                + rng.normal(0, 0.03, N_SUBJECTS)

    # an independent random stream per (cohort, subject), so schemes are comparable
    rate = {name: [encode(w, int(n), np.random.RandomState([seed, i]))
                   for i, (w, n) in enumerate(zip(weights, n_vars))]
            for name, encode in SCHEMES.items()}

    return dict(types=types, weights=weights, n_vars=n_vars,
                behaviour=behaviour, rate=rate)

## 2. The three dissimilarities

Both measures operate on the same PCA-reduced data with the same number of components,
and the ridge penalty is chosen by inner cross-validation, so the comparison is not
stacked in favour of either.

In [3]:
def to_components(rate):
    """Reduce every subject to the same number of PCs.  PCA also centres each
    neuron across conditions, which is the preprocessing both measures assume."""
    return [PCA(N_PCS).fit_transform(r.T) for r in rate]


def procrustes_distance(X, Y):
    """Rotation-invariant shape distance: symmetric and satisfies the triangle
    inequality, i.e. a proper metric."""
    metric = LinearMetric(alpha=1, center_columns=True, score_method="euclidean")
    metric.fit(X, Y)
    return metric.score(X, Y)


def predictivity_r2(X, Y):
    """Cross-validated linear predictivity of Y from X, held out over conditions.
    Asymmetric: how well X predicts Y is not how well Y predicts X."""
    scores = []
    for train, test in KFold(5, shuffle=True, random_state=0).split(X):
        model = RidgeCV(alphas=RIDGE_ALPHAS).fit(X[train], Y[train])
        scores.append(r2_score(Y[test], model.predict(X[test]),
                               multioutput="variance_weighted"))
    return float(np.mean(scores))


def dissimilarity_matrices(rate):
    """The three subject x subject dissimilarities being compared."""
    components = to_components(rate)
    n = len(components)
    procrustes = np.zeros((n, n))
    predictivity = np.zeros((n, n))
    for i in range(n):
        for j in range(n):
            if i < j:
                d = procrustes_distance(components[i], components[j])
                procrustes[i, j] = procrustes[j, i] = d
            if i != j:
                predictivity[i, j] = 1 - max(
                    predictivity_r2(components[i], components[j]), 0)
    return {"procrustes": procrustes,
            "predictivity": predictivity,
            "symmetrized": (predictivity + predictivity.T) / 2}


def asymmetry(dissimilarity):
    """mean |D - D.T| / mean D.  Zero for any symmetric dissimilarity."""
    off = ~np.eye(len(dissimilarity), dtype=bool)
    return float(np.abs(dissimilarity - dissimilarity.T)[off].mean()
                 / dissimilarity[off].mean())

## 3. Predicting behaviour from position in the space

Leave-one-out nearest-neighbour regression. With an asymmetric matrix, "the neighbours
of subject $i$" is not a well-defined set — row $i$ says who $i$ predicts well, column
$i$ says who predicts $i$ well, and there is no principled reason to prefer either.

In [4]:
def knn_predict(dissimilarity, behaviour, k=N_NEIGHBOURS):
    """Predict each subject's behaviour by averaging its k nearest neighbours."""
    masked = dissimilarity.astype(float).copy()
    np.fill_diagonal(masked, np.inf)          # a subject is never its own neighbour
    return np.array([behaviour[np.argsort(masked[i])[:k]].mean()
                     for i in range(len(masked))])


def evaluate(cohort):
    """R2 of the behaviour prediction for every (scheme, dissimilarity) pair."""
    return {scheme: {name: r2_score(cohort["behaviour"],
                                    knn_predict(D, cohort["behaviour"]))
                     for name, D in dissimilarity_matrices(rate).items()}
            for scheme, rate in cohort["rate"].items()}


def evaluate_seed(seed):
    return evaluate(make_cohort(seed))

## 4. One cohort

In [5]:
cohort = make_cohort(0)
matrices = {scheme: dissimilarity_matrices(rate)
            for scheme, rate in cohort["rate"].items()}

print(f"{'':<22}" + "".join(f"{s:>18}" for s in SCHEMES))
for name in METRICS:
    row = "".join(f"{r2_score(cohort['behaviour'], knn_predict(matrices[s][name], cohort['behaviour'])):>+18.2f}"
                  for s in SCHEMES)
    print(f"{name:<22}" + row)

print()
for scheme in SCHEMES:
    print(f"{scheme:<18} asymmetry: predictivity {asymmetry(matrices[scheme]['predictivity']):.2f}, "
          f"Procrustes {asymmetry(matrices[scheme]['procrustes']):.0e}")

                              segregated      linear mixed   nonlinear mixed
procrustes                         +0.98             +0.98             +0.98
predictivity                       -0.46             -0.42             -0.19
symmetrized                        +0.55             -0.07             +0.43

segregated         asymmetry: predictivity 1.49, Procrustes 0e+00
linear mixed       asymmetry: predictivity 0.87, Procrustes 0e+00
nonlinear mixed    asymmetry: predictivity 1.37, Procrustes 0e+00


## 5. Fifty cohorts

The same functions, resampled — so the numbers above are exactly the `seed = 0` entry
of the summary below.

In [6]:
results = Parallel(n_jobs=-2)(delayed(evaluate_seed)(s) for s in range(N_SEEDS))

R2 = {scheme: {name: np.array([r[scheme][name] for r in results])
               for name in METRICS}
      for scheme in SCHEMES}

# the single cohort above must be seed 0 of this sweep
for scheme in SCHEMES:
    for name in METRICS:
        expected = r2_score(cohort["behaviour"],
                            knn_predict(matrices[scheme][name], cohort["behaviour"]))
        assert abs(R2[scheme][name][0] - expected) < 1e-9, (scheme, name)

print(f"{N_SEEDS} cohorts, mean +- sd")
print(f"{'':<22}" + "".join(f"{s:>22}" for s in SCHEMES))
for name in METRICS:
    print(f"{name:<22}" + "".join(
        f"{R2[s][name].mean():>+15.2f} +-{R2[s][name].std():<5.2f}" for s in SCHEMES))

50 cohorts, mean +- sd
                                  segregated          linear mixed       nonlinear mixed
procrustes                      +0.97 +-0.01           +0.97 +-0.01           +0.97 +-0.01 
predictivity                    -0.40 +-0.27           -0.61 +-0.19           -0.19 +-0.42 
symmetrized                     +0.31 +-0.16           -0.15 +-0.28           +0.21 +-0.21 


## 6. The figure

In [7]:
# Same figure settings as the other analysis notebooks in this repository
plt.rcParams.update({"text.usetex": False, "svg.fonttype": "none", "pdf.fonttype": 42,
                     "font.family": "sans-serif", "font.sans-serif": ["Arial"],
                     "mathtext.fontset": "custom", "mathtext.rm": "Arial",
                     "mathtext.it": "Arial:italic", "mathtext.bf": "Arial:bold",
                     "axes.unicode_minus": False,
                     "font.size": 8, "axes.titlesize": 8, "axes.linewidth": 0.7})

STIMULUS_CMAP = sns.color_palette("husl", N_STIMULI)   # colour by stimulus
VIEW = np.random.RandomState(111)                      # random 3-D viewing angles
EXAMPLE_SCHEME = "linear mixed"    # every neuron is stimulus-tuned, so all contribute

# The top rows show only the stimulus conditions.  There the other variables sit at
# baseline, so their contribution is exactly zero and this is pure stimulus tuning.
# The dissimilarities themselves use all N_CONDITIONS.
tuning = [r[:, STIMULUS_ONLY] for r in cohort["rate"][EXAMPLE_SCHEME]]
behaviour = cohort["behaviour"]
# one example per type, matched on n_vars so the panel isolates tuning width
examples = [np.where(cohort["types"] == t)[0][np.argmin(cohort["n_vars"][cohort["types"] == t])]
            for t in range(len(TUNING_TYPES))]
lim = (behaviour.min() - 0.4, behaviour.max() + 0.4)

fig = plt.figure(figsize=(7.4, 6.6))
grid = fig.add_gridspec(3, 6, height_ratios=[1.9, 0.8, 2.4], hspace=0.7, wspace=0.9)

for col, (subject, title) in enumerate(zip(examples, TUNING_TYPES)):

    # --- the neural manifold
    ax = fig.add_subplot(grid[0, 2 * col:2 * col + 2], projection="3d")
    rotation = np.linalg.qr(VIEW.randn(3, 3))[0]
    x, y, z = rotation @ PCA(3).fit_transform(tuning[subject].T).T
    ax.plot(np.r_[x, x[0]], np.r_[y, y[0]], zs=np.r_[z, z[0]],
            lw=0.6, color="0.6", zorder=0)                       # close the loop
    ax.scatter(x, y, zs=z, c=STIMULUS_CMAP, lw=0, s=12)
    ax.plot(x, y, zs=ax.get_zlim()[0], lw=3, color="k", alpha=0.3)   # floor shadow
    ax.axis("off")
    ax.set_box_aspect(None, zoom=1.15)          # above ~1.2 the manifold is clipped
    ax.set_title(f"{title}\nbehaviour = {behaviour[subject]:.2f}")

    # --- example tuning curves, coloured by preferred stimulus
    ax = fig.add_subplot(grid[1, 2 * col:2 * col + 2])
    responsive = np.where(np.ptp(tuning[subject], axis=1) > 1e-9)[0]
    by_preference = responsive[np.argsort(tuning[subject][responsive].argmax(axis=1))]
    for neuron in by_preference[::max(len(by_preference) // 7, 1)]:
        ax.plot(STIMULI, tuning[subject][neuron], lw=0.9,
                color=STIMULUS_CMAP[tuning[subject][neuron].argmax()])
    ax.set_xticks([-np.pi, 0, np.pi], [r"$-\pi$", "0", r"$\pi$"])
    ax.set_yticks([])
    ax.set_xlabel("stimulus")
    if col == 0:
        ax.set_ylabel("firing rate")
    sns.despine(ax=ax, left=True)

# --- behaviour predicted from geometry, this cohort
ax = fig.add_subplot(grid[2, 0:3])
ax.plot(lim, lim, "-", lw=0.7, color="0.7", zorder=0)
for line, name in enumerate(reversed(METRICS)):        # Procrustes drawn last, on top
    predicted = knn_predict(matrices[EXAMPLE_SCHEME][name], behaviour)
    ax.scatter(behaviour, predicted, s=13, edgecolors="none",
               color=METRIC_COLOR[name])
    ax.text(0.02, 0.97 - 0.085 * line,
            f"{METRIC_LABEL[name]} ($R^2$ = {r2_score(behaviour, predicted):.2f})",
            transform=ax.transAxes, va="top", fontsize=6.5, color=METRIC_COLOR[name])
ax.set(xlim=lim, ylim=lim)
ax.set_aspect("equal")
ax.set_xlabel("true behaviour")
ax.set_ylabel(f"predicted (kNN, $k$={N_NEIGHBOURS})")
ax.set_title(f"one cohort, {EXAMPLE_SCHEME}")
sns.despine(ax=ax)

# --- across cohorts
ax = fig.add_subplot(grid[2, 3:6])
for group, scheme in enumerate(SCHEMES):
    for offset, name in enumerate(METRICS):
        values = R2[scheme][name]
        position = group * (len(METRICS) + 1) + offset + 1
        body = ax.violinplot([values], positions=[position], showextrema=False,
                             widths=0.85)["bodies"][0]
        body.set_facecolor(METRIC_COLOR[name])
        ax.plot([position - 0.38, position + 0.38], [values.mean()] * 2, "-",
                lw=1.6, color=METRIC_COLOR[name])
ax.axhline(0, ls=":", lw=0.7, color="0.4")
ax.set_xticks([2, 6, 10], ["segregated", "linear\nmixed", "nonlinear\nmixed"])
ax.set_xlim(0, 12)
ax.set_ylabel("$R^2$")
ax.set_box_aspect(1)
ax.set_title(f"{N_SEEDS} cohorts")
ax.legend(handles=[plt.Line2D([], [], lw=4, color=METRIC_COLOR[n],
                              label=METRIC_LABEL[n]) for n in METRICS],
          fontsize=6, frameon=False, loc="lower left", handlelength=1.2,
          handletextpad=0.4, labelspacing=0.2)
sns.despine(ax=ax)

# vector output; text stays editable in both formats
fig.savefig("figure_metric_vs_regression.pdf", transparent=True, bbox_inches="tight")
fig.savefig("figure_metric_vs_regression.svg", transparent=True, bbox_inches="tight")
plt.show()

findfont: Failed to find font weight normal, now using 0.


/var/folders/6w/snfd5vc50jb775fq8364m1d80000gn/T/ipykernel_35544/3077317558.py:97: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()
